# Feature Engineering

Handling missing data: fill with median

Random state: 42

## Imports and loading data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')
 
print("Imports successful")
 
# Load the EDA output
df = pd.read_csv('../EDA/mlb_pitches_eda.csv')
print(f"Loaded {len(df)} pitches")

Imports successful
Loaded 600414 pitches


## Data Cleaning
Just cleaning out rarer pitch types

In [2]:
# Pitch types with <20 samples are noise/data errors
rare_pitches = ['SV', 'CS', 'PO', 'UN', 'FA']
 
print(f"\nOriginal pitch types: {len(df['pitch_type'].unique())}")
print(f"Pitch counts before filtering:")
print(df['pitch_type'].value_counts())
 
# Filter
df = df[~df['pitch_type'].isin(rare_pitches)].copy()
 
print(f"\nRare pitches removed: {rare_pitches}")
print(f"New pitch types: {len(df['pitch_type'].unique())}")
print(f"Pitch counts after filtering:")
print(df['pitch_type'].value_counts())
 
print(f"\nDataset size after cleaning: {len(df)} pitches")


Original pitch types: 17
Pitch counts before filtering:
pitch_type
FF    183636
SI     99870
SL     79376
CH     67302
ST     50012
FC     47752
CU     38290
FS     19437
KC      9600
SV      2660
EP      1016
FA       648
FO       401
KN       247
CS       122
PO        34
UN        11
Name: count, dtype: int64

Rare pitches removed: ['SV', 'CS', 'PO', 'UN', 'FA']
New pitch types: 12
Pitch counts after filtering:
pitch_type
FF    183636
SI     99870
SL     79376
CH     67302
ST     50012
FC     47752
CU     38290
FS     19437
KC      9600
EP      1016
FO       401
KN       247
Name: count, dtype: int64

Dataset size after cleaning: 596939 pitches


## Features 1 and 2: Handedness

Reading pitcher handedness and batter handedness. Left-Left/Right-Right favors the pitcher, Left-Right/Right-Left favors the batter.

In [3]:
# Pitcher handedness: p_throws is L or R
df['pitcher_hand'] = df['p_throws'].map({'L': 0, 'R': 1})
print(f"Pitcher handedness distribution:")
print(df['p_throws'].value_counts())
print(f"  - Encoded as: L=0, R=1")
 
# Batter handedness: stand is L, R, or S (switch)
df['batter_hand'] = df['stand'].map({'L': 0, 'R': 1, 'S': 0.5})
print(f"\nBatter handedness distribution:")
print(df['stand'].value_counts())
print(f"  - Encoded as: L=0, R=1, S=0.5")
 
# Handedness matchup (4 categories)
# TODO: Data analysis on whether switch hitters always choose the opposite side of the pitcher to a significant degree. 
# If so, we can treat switch hitters as the opposite side of the pitcher.
def encode_matchup(row):
    p = row['p_throws']
    b = row['stand']
    matchups = {
        ('L', 'L'): 0,  # Left on Left
        ('L', 'R'): 1,  # Left on Right
        ('L', 'S'): 2,  # Left on Switch
        ('R', 'L'): 3,  # Right on Left
        ('R', 'R'): 4,  # Right on Right
        ('R', 'S'): 5,  # Right on Switch
    }
    return matchups.get((p, b), np.nan)
 
df['handedness_matchup'] = df.apply(encode_matchup, axis=1)
print(f"\nHandedness matchup distribution:")
print(df['handedness_matchup'].value_counts().sort_index())

Pitcher handedness distribution:
p_throws
R    421348
L    175591
Name: count, dtype: int64
  - Encoded as: L=0, R=1

Batter handedness distribution:
stand
R    303262
L    293677
Name: count, dtype: int64
  - Encoded as: L=0, R=1, S=0.5

Handedness matchup distribution:
handedness_matchup
0     58798
1    116793
3    234879
4    186469
Name: count, dtype: int64


## Feature 3: Count

In [4]:
# Already encoded in EDA as count_state; split back into separate features
df['balls'] = df['balls'].fillna(0).astype(int)
df['strikes'] = df['strikes'].fillna(0).astype(int)
 
print(f"Balls distribution:")
print(df['balls'].value_counts().sort_index())
print(f"\nStrikes distribution:")
print(df['strikes'].value_counts().sort_index())

Balls distribution:
balls
0    270608
1    176525
2    100201
3     49605
Name: count, dtype: int64

Strikes distribution:
strikes
0    238702
1    180646
2    177591
Name: count, dtype: int64


## Features 4 and 5: Outs and Inning

In [5]:
df['outs_when_up'] = df['outs_when_up'].fillna(0).astype(int)
df['inning'] = df['inning'].fillna(1).astype(int)
 
print(f"Outs distribution:")
print(df['outs_when_up'].value_counts().sort_index())
print(f"\nInning distribution (first 10 innings):")
print(df[df['inning'] <= 10]['inning'].value_counts().sort_index())

Outs distribution:
outs_when_up
0    205066
1    197649
2    194224
Name: count, dtype: int64

Inning distribution (first 10 innings):
inning
1     69216
2     66628
3     66762
4     66490
5     66597
6     67339
7     67889
8     68142
9     51063
10     5077
Name: count, dtype: int64


## Feature 6: Pitcher Arsenal

In [6]:
# For each pitcher, calculate % of each pitch type
pitcher_pitch_counts = df.groupby(['pitcher', 'pitch_type']).size().unstack(fill_value=0)
pitcher_pitch_totals = pitcher_pitch_counts.sum(axis=1)
pitcher_arsenal = pitcher_pitch_counts.div(pitcher_pitch_totals, axis=0) * 100
 
print(f"\nPitcher arsenal (first 5 pitchers, top 5 pitch types):")
print(pitcher_arsenal.iloc[:5, :5])

# Join back to main dataframe (each row gets their pitcher's arsenal)
df = df.merge(pitcher_arsenal, left_on='pitcher', right_index=True, how='left')
 
# Rename columns for clarity (FF_pct, SL_pct, etc.)
pitch_types = pitcher_arsenal.columns.tolist()
arsenal_cols = {pitch: f'{pitch}_pct' for pitch in pitch_types}
df = df.rename(columns=arsenal_cols)
 
print(f"\nAdded {len(arsenal_cols)} arsenal columns")
print(f"  - Example columns: {list(arsenal_cols.values())[:5]}")


Pitcher arsenal (first 5 pitchers, top 5 pitch types):
pitch_type         CH         CU   EP         FC         FF
pitcher                                                    
434378      10.000000  17.500000  0.0   0.000000  42.500000
445276       0.000000   0.000000  0.0  84.563758   0.000000
453286      16.000000  13.837838  0.0   4.756757  46.702703
455119       0.000000   5.645161  0.0  41.935484  26.209677
471911      30.894309   0.000000  0.0  15.447154  12.601626

Added 12 arsenal columns
  - Example columns: ['CH_pct', 'CU_pct', 'EP_pct', 'FC_pct', 'FF_pct']


## Feature 7: Batter's Career Stats

Focusing on K% and Contact%. Could introduce Whiff% in the future too?

In [7]:
# For each batter, calculate:
# - Strikeout rate: how often they strike out
# - Contact rate: how often they put the ball in play
 
# Count strikeouts: where 'events' contains 'strikeout'
batter_strikeouts = df[df['events'].str.contains('strikeout', case=False, na=False)].groupby('batter').size()
batter_at_bats = df.groupby('batter').size()  # Total pitches as proxy for at-bats (roughly)
 
batter_k_rate = (batter_strikeouts / batter_at_bats * 100).fillna(0)
 
print(f"Batter K% stats (first 10 batters):")
print(batter_k_rate.head(10))
print(f"\nOverall K% distribution:")
print(f"  Mean: {batter_k_rate.mean():.1f}%")
print(f"  Median: {batter_k_rate.median():.1f}%")
print(f"  Std: {batter_k_rate.std():.1f}%")
 
# Join back to main dataframe
df = df.merge(batter_k_rate.rename('batter_k_rate'), left_on='batter', right_index=True, how='left')
df['batter_k_rate'] = df['batter_k_rate'].fillna(df['batter_k_rate'].median())
 
print(f"\nAdded batter_k_rate")

Batter K% stats (first 10 batters):
batter
457705     6.232295
467793     7.547170
500743     2.904564
502054    11.666667
502671     6.000000
506702    10.404624
514888     5.625374
516782     6.230530
518595     7.346939
518692     4.029463
dtype: float64

Overall K% distribution:
  Mean: 6.3%
  Median: 6.0%
  Std: 2.7%

Added batter_k_rate


## Feature 8: Pitcher Workload Bucket

In [8]:
# Convert workload_bucket to numeric
workload_map = {'0-20': 0, '21-50': 1, '51-80': 2, '81+': 3}
df['workload_bucket_encoded'] = df['workload_bucket'].map(workload_map)
 
print(f"Workload bucket distribution:")
print(df['workload_bucket'].value_counts().sort_index())

Workload bucket distribution:
workload_bucket
0-20     284316
21-50    162521
51-80    112419
81+       37683
Name: count, dtype: int64


## Version 1:
Everything not mentioned as version 1 is version 0, or the initial working stuff. From here on out I'm going to be adding new stuff to play around with.

- Workload Bucket Validation
- Runners on Base and Game State (This one failed, since I couldn't get an O(n) solution to match the ABs)
- Recent Pitcher Performance

## (v1) Workload Bucket Validation

In [9]:
workload_counts = df['workload_bucket_encoded'].value_counts().sort_index()
bucket_names = {0: '0-20', 1: '21-50', 2: '51-80', 3: '80+'}
 
print(f"\nWorkload bucket sizes:")
all_sufficient = True
for bucket_id, count in workload_counts.items():
    bucket_name = bucket_names.get(bucket_id, 'Unknown')
    status = 'SUFFICIENT' if count >= 50 else 'SPARSE'
    print(f"  {bucket_name:6s}: {count:8,} {status}")
    if count < 50:
        all_sufficient = False
 
if all_sufficient:
    print(f"\nAll workload buckets have >50 samples (sufficient for training)")
else:
    print(f"\nWARNING: Some workload buckets sparse (<50 samples)")


Workload bucket sizes:
  0-20  :  284,316 SUFFICIENT
  21-50 :  162,521 SUFFICIENT
  51-80 :  112,419 SUFFICIENT
  80+   :   37,683 SUFFICIENT

All workload buckets have >50 samples (sufficient for training)


## Feature 9: Target variable: Pitch Type

In [10]:
# Encode pitch types as integers
pitch_type_encoder = LabelEncoder()
df['pitch_type_encoded'] = pitch_type_encoder.fit_transform(df['pitch_type'])
 
print(f"Pitch types and encodings:")
for i, pitch in enumerate(pitch_type_encoder.classes_):
    count = (df['pitch_type'] == pitch).sum()
    pct = count / len(df) * 100
    print(f"  {i:2d}: {pitch:3s} ({count:6d} pitches, {pct:5.1f}%)")
 
# Save encoder for later
with open('pitch_type_encoder.pkl', 'wb') as f:
    pickle.dump(pitch_type_encoder, f)
print(f"\nSaved pitch_type_encoder to pickle")

Pitch types and encodings:
   0: CH  ( 67302 pitches,  11.3%)
   1: CU  ( 38290 pitches,   6.4%)
   2: EP  (  1016 pitches,   0.2%)
   3: FC  ( 47752 pitches,   8.0%)
   4: FF  (183636 pitches,  30.8%)
   5: FO  (   401 pitches,   0.1%)
   6: FS  ( 19437 pitches,   3.3%)
   7: KC  (  9600 pitches,   1.6%)
   8: KN  (   247 pitches,   0.0%)
   9: SI  ( 99870 pitches,  16.7%)
  10: SL  ( 79376 pitches,  13.3%)
  11: ST  ( 50012 pitches,   8.4%)

Saved pitch_type_encoder to pickle


## Feature 10 (v1): Runners on Base and Game State

In [11]:
# Runners on Base (binary flags)
df['runner_on_1b'] = df['on_1b'].fillna(0).astype(int)
df['runner_on_2b'] = df['on_2b'].fillna(0).astype(int)
df['runner_on_3b'] = df['on_3b'].fillna(0).astype(int)
 
# Bases Loaded (all three)
df['bases_loaded'] = ((df['runner_on_1b'] == 1) & 
                       (df['runner_on_2b'] == 1) & 
                       (df['runner_on_3b'] == 1)).astype(int)
 
# Score Differential
df['score_diff'] = df['home_score'] - df['away_score']
 
print(f"\nRunner distribution:")
print(f"  Runners on 1B: {df['runner_on_1b'].sum():,} ({df['runner_on_1b'].mean()*100:.1f}%)")
print(f"  Runners on 2B: {df['runner_on_2b'].sum():,} ({df['runner_on_2b'].mean()*100:.1f}%)")
print(f"  Runners on 3B: {df['runner_on_3b'].sum():,} ({df['runner_on_3b'].mean()*100:.1f}%)")
print(f"  Bases loaded: {df['bases_loaded'].sum():,} ({df['bases_loaded'].mean()*100:.1f}%)")
 
print(f"\nScore differential: {df['score_diff'].min():.0f} to {df['score_diff'].max():.0f}")
print(f"  Mean: {df['score_diff'].mean():.2f}")


Runner distribution:
  Runners on 1B: 124,991,132,240 (20938677.5%)
  Runners on 2B: 76,445,802,238 (12806300.5%)
  Runners on 3B: 38,414,388,287 (6435228.4%)
  Bases loaded: 0 (0.0%)

Score differential: -21 to 22
  Mean: -0.13


## Selecting Features for XGBoost

In [12]:
# Version 0 features
feature_cols_v0 = [
    'balls',
    'strikes',
    'pitcher_hand',
    'batter_hand',
    'handedness_matchup',
    'outs_when_up',
    'inning',
    'workload_bucket_encoded',
    'batter_k_rate',
]
 
# Version 1 features
feature_cols_v1 = [
    'runner_on_1b',
    'runner_on_2b',
    'runner_on_3b',
    'bases_loaded',
    'score_diff',
]
 
# Combine all features
feature_cols = feature_cols_v0 + feature_cols_v1
 
# Add arsenal columns
feature_cols.extend([col for col in df.columns if col.endswith('_pct')])
 
print(f"\nXGBoost features ({len(feature_cols)} total):")
print(f"\n  VERSION 0 ({len(feature_cols_v0)} features):")
for i, col in enumerate(feature_cols_v0, 1):
    print(f"    {i:2d}. {col}")
 
print(f"\n  VERSION 1 NEW ({len(feature_cols_v1)} features):")
for i, col in enumerate(feature_cols_v1, len(feature_cols_v0) + 1):
    print(f"    {i:2d}. {col} [NEW]")
 
arsenal_count = len([c for c in feature_cols if c.endswith('_pct')])
print(f"\n  PITCHER ARSENAL ({arsenal_count} pitch types):")
for col in sorted([c for c in feature_cols if c.endswith('_pct')])[:5]:
    print(f"    - {col}")
if arsenal_count > 5:
    print(f"    ... and {arsenal_count - 5} more")
 
# Create feature matrix and target
X = df[feature_cols].copy()
y = df['pitch_type_encoded'].copy()
 
# Check for missing values
print(f"\nMissing values per feature:")
missing = X.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
    # Fill any remaining NaNs with median
    X = X.fillna(X.median())
    print("Filled with median values")
else:
    print("  None!")
 
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")


XGBoost features (26 total):

  VERSION 0 (9 features):
     1. balls
     2. strikes
     3. pitcher_hand
     4. batter_hand
     5. handedness_matchup
     6. outs_when_up
     7. inning
     8. workload_bucket_encoded
     9. batter_k_rate

  VERSION 1 NEW (5 features):
    10. runner_on_1b [NEW]
    11. runner_on_2b [NEW]
    12. runner_on_3b [NEW]
    13. bases_loaded [NEW]
    14. score_diff [NEW]

  PITCHER ARSENAL (12 pitch types):
    - CH_pct
    - CU_pct
    - EP_pct
    - FC_pct
    - FF_pct
    ... and 7 more

Missing values per feature:
  None!

Feature matrix shape: (596939, 26)
Target shape: (596939,)


## Train/Test Split

IMPORTANT: Holdout by pitcher

In [13]:
# Get unique pitchers
unique_pitchers = df['pitcher'].unique()
print(f"Total unique pitchers: {len(unique_pitchers)}")
 
# Split pitchers into train/test (80/20 by pitcher)
np.random.seed(42)
train_pitchers = np.random.choice(unique_pitchers, size=int(0.8 * len(unique_pitchers)), replace=False)
test_pitchers = np.setdiff1d(unique_pitchers, train_pitchers)
 
print(f"Train pitchers: {len(train_pitchers)}")
print(f"Test pitchers: {len(test_pitchers)}")
 
# Create train/test indices
train_idx = df['pitcher'].isin(train_pitchers)
test_idx = df['pitcher'].isin(test_pitchers)
 
X_train = X[train_idx].copy()
y_train = y[train_idx].copy()
X_test = X[test_idx].copy()
y_test = y[test_idx].copy()
 
print(f"\nTrain set: {len(X_train)} pitches from {train_idx.sum()} rows")
print(f"Test set: {len(X_test)} pitches from {test_idx.sum()} rows")
 
# Verify no pitcher leakage
train_pitcher_set = set(df[train_idx]['pitcher'].unique())
test_pitcher_set = set(df[test_idx]['pitcher'].unique())
print(f"\nPitcher leakage check: {len(train_pitcher_set & test_pitcher_set)} overlapping pitchers")
if len(train_pitcher_set & test_pitcher_set) == 0:
    print("No pitcher leakage. Train and test are isolated!")
else:
    print("WARNING: Pitcher leakage detected!")

Total unique pitchers: 828
Train pitchers: 662
Test pitchers: 166

Train set: 481704 pitches from 481704 rows
Test set: 115235 pitches from 115235 rows

Pitcher leakage check: 0 overlapping pitchers
No pitcher leakage. Train and test are isolated!


## Preparing data for LSTM

For LSTM, we need sequences of pitches, not individual pitches. For each at-bat, we create a sequence of last N pitches as input.

In [14]:
def create_sequences_for_lstm(df_subset, feature_cols, target_col='pitch_type_encoded', seq_length=5):
    sequences = []
    targets = []
    
    # Group by at-bat (pitcher_batter_pair + game_date)
    df_sorted = df_subset.sort_values(['game_date', 'pitcher', 'batter', 'pitch_number'])
    
    for (game_date, pitcher, batter), group in df_sorted.groupby(['game_date', 'pitcher', 'batter']):
        if len(group) < 2:  # Need at least 2 pitches per AB
            continue
        
        # Get feature sequences and targets
        group_features = group[feature_cols].values
        group_targets = group[target_col].values
        
        # Create sliding windows: each window is [past N pitches] -> next pitch
        for i in range(1, len(group)):  # Start from 1 (need at least 1 prior pitch)
            # Input: features of the current pitch (which includes count, outs, etc.). This is the context for predicting the next pitch.
            sequences.append(group_features[i])
            targets.append(group_targets[i])
    
    return np.array(sequences), np.array(targets)
 
print("Creating LSTM sequences...")
X_train_lstm, y_train_lstm = create_sequences_for_lstm(df[train_idx], feature_cols)
X_test_lstm, y_test_lstm = create_sequences_for_lstm(df[test_idx], feature_cols)
 
print(f"LSTM Train sequences: {X_train_lstm.shape}")
print(f"LSTM Test sequences: {X_test_lstm.shape}")

Creating LSTM sequences...
LSTM Train sequences: (402990, 26)
LSTM Test sequences: (95872, 26)


## Exports for model training

In [15]:
# Save XGBoost data
np.save('X_train.npy', X_train.values)
np.save('X_test.npy', X_test.values)
np.save('y_train.npy', y_train.values)
np.save('y_test.npy', y_test.values)
 
print("Saved XGBoost data:")
print(f"  X_train.npy ({X_train.shape})")
print(f"  X_test.npy ({X_test.shape})")
print(f"  y_train.npy ({y_train.shape})")
print(f"  y_test.npy ({y_test.shape})")
 
# Save LSTM data
np.save('X_train_lstm.npy', X_train_lstm)
np.save('X_test_lstm.npy', X_test_lstm)
np.save('y_train_lstm.npy', y_train_lstm)
np.save('y_test_lstm.npy', y_test_lstm)
 
print("Saved LSTM data:")
print(f"  X_train_lstm.npy ({X_train_lstm.shape})")
print(f"  X_test_lstm.npy ({X_test_lstm.shape})")
print(f"  y_train_lstm.npy ({y_train_lstm.shape})")
print(f"  y_test_lstm.npy ({y_test_lstm.shape})")
 
# Save test pitcher IDs for later evaluation
test_pitcher_ids = df[test_idx]['pitcher'].values
np.save('test_pitcher_ids.npy', test_pitcher_ids)
print("Saved test_pitcher_ids.npy")
 
# Save feature names for model interpretation
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print("Saved feature_names.pkl")

# Save pitch encoder
with open('pitch_type_encoder.pkl', 'wb') as f:
    pickle.dump(pitch_type_encoder, f)
print("Saved pitch_type_encoder.pkl")
 
# Save test dataframe info for analysis
test_df_info = df[test_idx][['game_date', 'pitcher', 'batter', 'pitch_type', 'pitch_type_encoded', 'balls', 'strikes']].copy()
test_df_info.to_csv('test_set_info.csv', index=False)
print("Saved test_set_info.csv")

Saved XGBoost data:
  X_train.npy ((481704, 26))
  X_test.npy ((115235, 26))
  y_train.npy ((481704,))
  y_test.npy ((115235,))
Saved LSTM data:
  X_train_lstm.npy ((402990, 26))
  X_test_lstm.npy ((95872, 26))
  y_train_lstm.npy ((402990,))
  y_test_lstm.npy ((95872,))
Saved test_pitcher_ids.npy
Saved feature_names.pkl
Saved pitch_type_encoder.pkl
Saved test_set_info.csv
